# Phase 7 — Evolving Fuzzy Reasoning System

Dự án: **An Evolving Fuzzy Reasoning System for Sensor Stream Anomaly Detection under Concept Drift**


In [1]:
# Phase 7.0 — Setup môi trường, nạp dữ liệu và khởi tạo Static Fuzzy Baseline Knowledge
from pathlib import Path
import pandas as pd
import numpy as np
import skfuzzy as fuzz

# 1. Nạp dữ liệu gốc và trích xuất Test stream (Phase 1.5)
data_path = Path("../data/ai4i2020.csv") if Path("../data/ai4i2020.csv").exists() else Path("data/ai4i2020.csv")
df = pd.read_csv(data_path)
test_df = df.iloc[8000:].copy()
stream_df = test_df.copy().reset_index(drop=True)

# 2. Universe of Discourse (Phase 2 & Phase 4)
universes = {
    "air_temp":     np.linspace(295.3, 304.5, 1000),
    "process_temp": np.linspace(305.7, 313.8, 1000),
    "rpm":          np.linspace(1168,  2886,  1000),
    "torque":       np.linspace(3.8,   76.2,  1000),
    "tool_wear":    np.linspace(0,     253,   1000),
    "anomaly":      np.linspace(0,     1,     1000),
}

# 3. Khởi tạo các hàm thuộc tính Static Fuzzy gốc đã đóng băng (Phase 2)
air_temp = universes["air_temp"]
air_low = fuzz.trapmf(air_temp, [295.3, 295.3, 298.0, 300.4])
air_medium = fuzz.trimf(air_temp, [298.0, 300.4, 302.5])
air_high = fuzz.trapmf(air_temp, [300.4, 302.5, 304.5, 304.5])

process_temp = universes["process_temp"]
process_low = fuzz.trapmf(process_temp, [305.7, 305.7, 308.0, 309.7])
process_medium = fuzz.trimf(process_temp, [308.0, 309.7, 311.5])
process_high = fuzz.trapmf(process_temp, [309.7, 311.5, 313.8, 313.8])

rpm = universes["rpm"]
rpm_low = fuzz.trapmf(rpm, [1168.0, 1168.0, 1350.0, 1500.0])
rpm_medium = fuzz.trimf(rpm, [1350.0, 1504.0, 1800.0])
rpm_high = fuzz.trapmf(rpm, [1600.0, 1880.0, 2886.0, 2886.0])

torque = universes["torque"]
torque_low = fuzz.trapmf(torque, [3.8, 3.8, 25.0, 40.0])
torque_medium = fuzz.trimf(torque, [25.0, 40.0, 55.0])
torque_high = fuzz.trapmf(torque, [40.0, 55.0, 76.2, 76.2])

tool_wear = universes["tool_wear"]
tool_wear_low = fuzz.trapmf(tool_wear, [0.0, 0.0, 54.0, 109.0])
tool_wear_medium = fuzz.trimf(tool_wear, [54.0, 109.0, 164.0])
tool_wear_high = fuzz.trapmf(tool_wear, [109.0, 164.0, 253.0, 253.0])

TAU_STATIC = 0.67

print("Setup Phase 7 hoàn tất: Dữ liệu và Static Fuzzy Knowledge Base đã sẵn sàng.")

# 4. Tạo các luồng Sudden Drift và Gradual Drift chuẩn mực (Phase 4)
SUDDEN_DRIFT_POINT = 1000
RPM_SHIFT = -150
TORQUE_SHIFT = 8
RPM_MIN = universes["rpm"].min()
RPM_MAX = universes["rpm"].max()
TORQUE_MIN = universes["torque"].min()
TORQUE_MAX = universes["torque"].max()

sudden_drift_stream = stream_df.copy()
after_drift = sudden_drift_stream.index >= SUDDEN_DRIFT_POINT
sudden_drift_stream.loc[after_drift, "Rotational speed [rpm]"] = (
    sudden_drift_stream.loc[after_drift, "Rotational speed [rpm]"] + RPM_SHIFT
).clip(RPM_MIN, RPM_MAX)
sudden_drift_stream.loc[after_drift, "Torque [Nm]"] = (
    sudden_drift_stream.loc[after_drift, "Torque [Nm]"] + TORQUE_SHIFT
).clip(TORQUE_MIN, TORQUE_MAX)

GRADUAL_DRIFT_START = 800
GRADUAL_DRIFT_END = 1200
gradual_drift_stream = stream_df.copy()
gradual_drift_stream['Rotational speed [rpm]'] = gradual_drift_stream['Rotational speed [rpm]'].astype(float)
gradual_drift_stream['Torque [Nm]'] = gradual_drift_stream['Torque [Nm]'].astype(float)
for i in gradual_drift_stream.index:
    if i < GRADUAL_DRIFT_START:
        alpha = 0.0
    elif i >= GRADUAL_DRIFT_END:
        alpha = 1.0
    else:
        alpha = (i - GRADUAL_DRIFT_START) / (GRADUAL_DRIFT_END - GRADUAL_DRIFT_START)

    gradual_drift_stream.loc[i, "Rotational speed [rpm]"] = np.clip(
        stream_df.loc[i, "Rotational speed [rpm]"] + alpha * RPM_SHIFT,
        RPM_MIN, RPM_MAX
    )
    gradual_drift_stream.loc[i, "Torque [Nm]"] = np.clip(
        stream_df.loc[i, "Torque [Nm]"] + alpha * TORQUE_SHIFT,
        TORQUE_MIN, TORQUE_MAX
    )

sudden_stream = sudden_drift_stream
gradual_stream = gradual_drift_stream

# 5. Cấu hình Static Mamdani FIS Engine đã đóng băng (Phase 2 & Phase 3 & Phase 6)
mf_params = {
    "air_temp": {
        "LOW":    ("trap", [295.3, 295.3, 298.0, 300.4]),
        "MEDIUM": ("tri",  [298.0, 300.4, 302.5]),
        "HIGH":   ("trap", [300.4, 302.5, 304.5, 304.5])
    },
    "process_temp": {
        "LOW":    ("trap", [305.7, 305.7, 308.0, 309.7]),
        "MEDIUM": ("tri",  [308.0, 309.7, 311.5]),
        "HIGH":   ("trap", [309.7, 311.5, 313.8, 313.8])
    },
    "rpm": {
        "LOW":    ("trap", [1168.0, 1168.0, 1350.0, 1500.0]),
        "MEDIUM": ("tri",  [1350.0, 1504.0, 1800.0]),
        "HIGH":   ("trap", [1600.0, 1880.0, 2886.0, 2886.0])
    },
    "torque": {
        "LOW":    ("trap", [3.8, 3.8, 25.0, 40.0]),
        "MEDIUM": ("tri",  [25.0, 40.0, 55.0]),
        "HIGH":   ("trap", [40.0, 55.0, 76.2, 76.2])
    },
    "tool_wear": {
        "LOW":    ("trap", [0.0, 0.0, 54.0, 109.0]),
        "MEDIUM": ("tri",  [54.0, 109.0, 164.0]),
        "HIGH":   ("trap", [109.0, 164.0, 253.0, 253.0])
    }
}

anomaly_universe = universes["anomaly"]
anomaly_mfs = {
    "LOW":    fuzz.trapmf(anomaly_universe, [0.0, 0.0, 0.25, 0.50]),
    "MEDIUM": fuzz.trimf( anomaly_universe, [0.25, 0.50, 0.75]),
    "HIGH":   fuzz.trapmf(anomaly_universe, [0.50, 0.75, 1.0, 1.0])
}

def eval_mf(x, mf_type, params):
    x = float(x)
    if mf_type == "tri":
        a, b, c = params
        if a < b and a <= x <= b:
            return (x - a) / (b - a)
        elif b < c and b <= x <= c:
            return (c - x) / (c - b)
        elif x == b:
            return 1.0
        return 0.0
    elif mf_type == "trap":
        a, b, c, d = params
        if x < a:
            return 1.0 if a == b else 0.0
        elif a <= x < b:
            return (x - a) / (b - a) if b > a else 1.0
        elif b <= x <= c:
            return 1.0
        elif c < x <= d:
            return (d - x) / (d - c) if d > c else 1.0
        else:
            return 1.0 if c == d else 0.0

def compute_memberships(sample):
    mapping = {
        "air_temp":     sample["Air temperature [K]"],
        "process_temp": sample["Process temperature [K]"],
        "rpm":          sample["Rotational speed [rpm]"],
        "torque":       sample["Torque [Nm]"],
        "tool_wear":    sample["Tool wear [min]"],
    }
    m = {}
    for var_name, val in mapping.items():
        m[var_name] = {}
        for term, (m_type, params) in mf_params[var_name].items():
            m[var_name][term] = eval_mf(val, m_type, params)
    return m

rules = [
    # HIGH anomaly — 4 rules
    ("R1", [("rpm", "LOW"), ("torque", "HIGH"), ("air_temp", "HIGH")], "HIGH"),
    ("R2", [("torque", "HIGH"), ("air_temp", "HIGH"), ("tool_wear", "HIGH")], "HIGH"),
    ("R3", [("rpm", "LOW"), ("torque", "HIGH"), ("process_temp", "HIGH")], "HIGH"),
    ("R4", [("rpm", "LOW"), ("torque", "HIGH"), ("tool_wear", "HIGH")], "HIGH"),

    # MEDIUM anomaly — 5 rules
    ("R5", [("rpm", "LOW"), ("torque", "HIGH")], "MEDIUM"),
    ("R6", [("rpm", "LOW"), ("air_temp", "HIGH")], "MEDIUM"),
    ("R7", [("torque", "HIGH"), ("air_temp", "HIGH")], "MEDIUM"),
    ("R8", [("torque", "HIGH"), ("tool_wear", "HIGH")], "MEDIUM"),
    ("R9", [("torque", "HIGH"), ("process_temp", "HIGH")], "MEDIUM"),

    # LOW anomaly — 3 rules
    ("R10", [("rpm", "MEDIUM"), ("torque", "MEDIUM")], "LOW"),
    ("R11", [("rpm", "MEDIUM"), ("torque", "LOW")], "LOW"),
    ("R12", [("rpm", "HIGH"), ("torque", "LOW")], "LOW"),
]

def mamdani_inference(sample, return_details=False):
    mu = compute_memberships(sample)
    rule_activations = {}
    implied_outputs = []
    
    for r_id, antecedents, consequent in rules:
        alpha = min(mu[var][term] for var, term in antecedents)
        rule_activations[r_id] = alpha
        implied_mf = np.fmin(alpha, anomaly_mfs[consequent])
        implied_outputs.append(implied_mf)
        
    aggregated_mf = np.zeros_like(anomaly_universe)
    for mf in implied_outputs:
        aggregated_mf = np.fmax(aggregated_mf, mf)
        
    if np.sum(aggregated_mf) == 0:
        score = 0.0
    else:
        score = fuzz.defuzz(anomaly_universe, aggregated_mf, "centroid")
        
    if return_details:
        return score, rule_activations, mu, aggregated_mf
    return score


Setup Phase 7 hoàn tất: Dữ liệu và Static Fuzzy Knowledge Base đã sẵn sàng.


## Phase 7.1 — Freeze Static Baseline / Create Evolving Copy


In [2]:
print("=== PHASE 7.1 — FREEZE STATIC BASELINE ===")

# Copy the final Static Fuzzy MF parameters
evolving_mfs = {
    "air": {
        "low": air_low.copy(),
        "medium": air_medium.copy(),
        "high": air_high.copy(),
    },
    "process": {
        "low": process_low.copy(),
        "medium": process_medium.copy(),
        "high": process_high.copy(),
    },
    "rpm": {
        "low": rpm_low.copy(),
        "medium": rpm_medium.copy(),
        "high": rpm_high.copy(),
    },
    "torque": {
        "low": torque_low.copy(),
        "medium": torque_medium.copy(),
        "high": torque_high.copy(),
    },
    "tool_wear": {
        "low": tool_wear_low.copy(),
        "medium": tool_wear_medium.copy(),
        "high": tool_wear_high.copy(),
    },
}

print("\nStatic baseline: FROZEN")
print("Evolving copy: CREATED")

# Verify that all copied membership functions initially match
checks = []

for variable in evolving_mfs:
    for term in evolving_mfs[variable]:
        static_mf = globals()[f"{variable}_{term}"]
        evolving_mf = evolving_mfs[variable][term]

        checks.append(np.array_equal(static_mf, evolving_mf))

print(f"\nMF copy checks passed: {sum(checks)}/{len(checks)}")
print(f"All initial MFs identical: {all(checks)}")

print("\nProtected Static Parameters:")
print("- Original membership functions")
print("- Original 12-rule structure")
print("- TAU_STATIC = 0.67")

print("\nEvolving Parameters:")
print("- Separate RPM membership functions")
print("- Separate Torque membership functions")


=== PHASE 7.1 — FREEZE STATIC BASELINE ===

Static baseline: FROZEN
Evolving copy: CREATED

MF copy checks passed: 15/15
All initial MFs identical: True

Protected Static Parameters:
- Original membership functions
- Original 12-rule structure
- TAU_STATIC = 0.67

Evolving Parameters:
- Separate RPM membership functions
- Separate Torque membership functions


## Phase 7.2 — Define Adaptation Buffer


In [3]:
print("=== PHASE 7.2 — DEFINE ADAPTATION BUFFER ===")

ADAPTATION_WINDOW = 200

sudden_detection = 1183
gradual_detection = 1247

sudden_adapt_start = sudden_detection + 1
gradual_adapt_start = gradual_detection + 1

sudden_adapt_end = sudden_adapt_start + ADAPTATION_WINDOW
gradual_adapt_end = gradual_adapt_start + ADAPTATION_WINDOW

print("\nAdaptation protocol:")
print(f"Window size: {ADAPTATION_WINDOW} samples")
print("Adaptation starts AFTER ADWIN detection.")
print("Adaptation data is not used to evaluate the same sample.")

print("\nSudden Drift")
print(f"Detection index: {sudden_detection}")
print(f"Adaptation range: {sudden_adapt_start} -> {sudden_adapt_end - 1}")
print(f"Adaptation samples: {sudden_adapt_end - sudden_adapt_start}")

print("\nGradual Drift")
print(f"Detection index: {gradual_detection}")
print(f"Adaptation range: {gradual_adapt_start} -> {gradual_adapt_end - 1}")
print(f"Adaptation samples: {gradual_adapt_end - gradual_adapt_start}")

print("\nLeakage constraints:")
print("- No adaptation before drift detection.")
print("- Current sample must be evaluated before adaptation.")
print("- No future samples are used.")


=== PHASE 7.2 — DEFINE ADAPTATION BUFFER ===

Adaptation protocol:
Window size: 200 samples
Adaptation starts AFTER ADWIN detection.
Adaptation data is not used to evaluate the same sample.

Sudden Drift
Detection index: 1183
Adaptation range: 1184 -> 1383
Adaptation samples: 200

Gradual Drift
Detection index: 1247
Adaptation range: 1248 -> 1447
Adaptation samples: 200

Leakage constraints:
- No adaptation before drift detection.
- Current sample must be evaluated before adaptation.
- No future samples are used.


## Phase 7.3 — Inspect Adaptation Buffer


In [4]:
print("=== PHASE 7.3 — INSPECT ADAPTATION BUFFER ===")

sensor_columns = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]

def inspect_adaptation_buffer(stream, detection_index, window_size, name):
    buffer_start = detection_index + 1
    buffer_end = buffer_start + window_size

    baseline = stream.iloc[:detection_index + 1]
    buffer = stream.iloc[buffer_start:buffer_end]

    print(f"\n{name}")
    print(f"Detection index: {detection_index}")
    print(f"Buffer range: {buffer_start} -> {buffer_end - 1}")
    print(f"Buffer size: {len(buffer)}")

    print("\nSensor means:")
    for col in sensor_columns:
        baseline_mean = baseline[col].mean()
        buffer_mean = buffer[col].mean()
        delta = buffer_mean - baseline_mean

        print(
            f"{col}: "
            f"baseline={baseline_mean:.4f}, "
            f"buffer={buffer_mean:.4f}, "
            f"delta={delta:+.4f}"
        )


inspect_adaptation_buffer(
    sudden_stream,
    sudden_detection,
    ADAPTATION_WINDOW,
    "Sudden Drift"
)

inspect_adaptation_buffer(
    gradual_stream,
    gradual_detection,
    ADAPTATION_WINDOW,
    "Gradual Drift"
)


=== PHASE 7.3 — INSPECT ADAPTATION BUFFER ===

Sudden Drift
Detection index: 1183
Buffer range: 1184 -> 1383
Buffer size: 200

Sensor means:
Air temperature [K]: baseline=298.2779, buffer=298.1310, delta=-0.1469
Process temperature [K]: baseline=309.4181, buffer=308.8940, delta=-0.5241
Rotational speed [rpm]: baseline=1508.6225, buffer=1395.1200, delta=-113.5025
Torque [Nm]: baseline=41.3268, buffer=47.3015, delta=+5.9747
Tool wear [min]: baseline=105.3117, buffer=97.9650, delta=-7.3467

Gradual Drift
Detection index: 1247
Buffer range: 1248 -> 1447
Buffer size: 200

Sensor means:
Air temperature [K]: baseline=298.2632, buffer=298.1055, delta=-0.1577
Process temperature [K]: baseline=309.3926, buffer=308.7440, delta=-0.6486
Rotational speed [rpm]: baseline=1504.1442, buffer=1389.3750, delta=-114.7692
Torque [Nm]: baseline=41.5861, buffer=48.0530, delta=+6.4669
Tool wear [min]: baseline=104.8638, buffer=99.9900, delta=-4.8738


## Phase 7.4 — Estimate MF Shift from Pre-Drift Baseline


In [5]:
print("=== PHASE 7.4 — ESTIMATE MF SHIFT FROM PRE-DRIFT BASELINE ===")

def estimate_mf_shift(stream, pre_drift_end, detection_index, window_size, name):
    pre_drift = stream.iloc[:pre_drift_end + 1]

    buffer_start = detection_index + 1
    buffer_end = buffer_start + window_size
    buffer = stream.iloc[buffer_start:buffer_end]

    print(f"\n{name}")
    print(f"Pre-drift baseline: 0 -> {pre_drift_end}")
    print(f"Detection index: {detection_index}")
    print(f"Adaptation buffer: {buffer_start} -> {buffer_end - 1}")

    for col in ["Rotational speed [rpm]", "Torque [Nm]"]:
        baseline_mean = pre_drift[col].mean()
        buffer_mean = buffer[col].mean()
        delta = buffer_mean - baseline_mean

        print(
            f"{col}: "
            f"pre_drift_mean={baseline_mean:.4f}, "
            f"buffer_mean={buffer_mean:.4f}, "
            f"estimated_shift={delta:+.4f}"
        )


estimate_mf_shift(
    sudden_stream,
    pre_drift_end=999,
    detection_index=sudden_detection,
    window_size=ADAPTATION_WINDOW,
    name="Sudden Drift"
)

estimate_mf_shift(
    gradual_stream,
    pre_drift_end=799,
    detection_index=gradual_detection,
    window_size=ADAPTATION_WINDOW,
    name="Gradual Drift"
)


=== PHASE 7.4 — ESTIMATE MF SHIFT FROM PRE-DRIFT BASELINE ===

Sudden Drift
Pre-drift baseline: 0 -> 999
Detection index: 1183
Adaptation buffer: 1184 -> 1383
Rotational speed [rpm]: pre_drift_mean=1528.0220, buffer_mean=1395.1200, estimated_shift=-132.9020
Torque [Nm]: pre_drift_mean=40.2530, buffer_mean=47.3015, estimated_shift=+7.0485

Gradual Drift
Pre-drift baseline: 0 -> 799
Detection index: 1247
Adaptation buffer: 1248 -> 1447
Rotational speed [rpm]: pre_drift_mean=1527.5612, buffer_mean=1389.3750, estimated_shift=-138.1862
Torque [Nm]: pre_drift_mean=40.1745, buffer_mean=48.0530, estimated_shift=+7.8785


## Phase 7.5 — Create Adapted Fuzzy Memberships


In [6]:
print("=== PHASE 7.5 — CREATE ADAPTED FUZZY MEMBERSHIPS (FIXED) ===")

# Bridge: Flatten evolving_mfs from nested format to flat keys for direct term access
if "air_low" not in evolving_mfs and "air" in evolving_mfs:
    evolving_mfs = {
        f"{var}_{term}": evolving_mfs[var][term]
        for var in evolving_mfs
        for term in evolving_mfs[var]
    }

def shift_mf(mf, shift, universe):
    # Shift the MF horizontally by `shift`.
    # Positive shift  -> MF moves right
    # Negative shift  -> MF moves left
    shifted_mf = np.interp(
        universe - shift,
        universe,
        mf,
        left=0.0,
        right=0.0
    )

    return shifted_mf


def create_adapted_mfs(base_mfs, rpm_shift, torque_shift):
    adapted = {
        name: mf.copy()
        for name, mf in base_mfs.items()
    }

    # Adapt only RPM MFs
    for name in ["rpm_low", "rpm_medium", "rpm_high"]:
        adapted[name] = shift_mf(
            base_mfs[name],
            rpm_shift,
            rpm
        )

    # Adapt only Torque MFs
    for name in ["torque_low", "torque_medium", "torque_high"]:
        adapted[name] = shift_mf(
            base_mfs[name],
            torque_shift,
            torque
        )

    return adapted


# Estimated shifts from Cell 7.4
sudden_rpm_shift = -132.9020
sudden_torque_shift = +7.0485

gradual_rpm_shift = -138.1862
gradual_torque_shift = +7.8785


# Create scenario-specific evolving fuzzy knowledge
sudden_adapted_mfs = create_adapted_mfs(
    evolving_mfs,
    rpm_shift=sudden_rpm_shift,
    torque_shift=sudden_torque_shift
)

gradual_adapted_mfs = create_adapted_mfs(
    evolving_mfs,
    rpm_shift=gradual_rpm_shift,
    torque_shift=gradual_torque_shift
)


# Protected MFs must remain unchanged
protected_mfs = [
    "air_low", "air_medium", "air_high",
    "process_low", "process_medium", "process_high",
    "tool_wear_low", "tool_wear_medium", "tool_wear_high",
]

for name in protected_mfs:
    assert np.array_equal(
        sudden_adapted_mfs[name],
        evolving_mfs[name]
    )
    assert np.array_equal(
        gradual_adapted_mfs[name],
        evolving_mfs[name]
    )

print("Protected Air/Process/Tool-wear MFs unchanged: PASS")
print("RPM and Torque MFs adapted: PASS")
print("Rule structure unchanged: 12 rules")
print(f"Sudden RPM shift: {sudden_rpm_shift:+.4f}")
print(f"Sudden Torque shift: {sudden_torque_shift:+.4f}")
print(f"Gradual RPM shift: {gradual_rpm_shift:+.4f}")
print(f"Gradual Torque shift: {gradual_torque_shift:+.4f}")


=== PHASE 7.5 — CREATE ADAPTED FUZZY MEMBERSHIPS (FIXED) ===
Protected Air/Process/Tool-wear MFs unchanged: PASS
RPM and Torque MFs adapted: PASS
Rule structure unchanged: 12 rules
Sudden RPM shift: -132.9020
Sudden Torque shift: +7.0485
Gradual RPM shift: -138.1862
Gradual Torque shift: +7.8785


## Phase 7.6 — Verify Adapted MF Geometry


In [7]:
print("=== PHASE 7.6 — VERIFY ADAPTED MF GEOMETRY ===")


def mf_peak_x(mf, universe):
    return universe[np.argmax(mf)]


def mf_centroid_x(mf, universe):
    area = np.trapezoid(mf, universe)

    if area == 0:
        return np.nan

    return np.trapezoid(universe * mf, universe) / area


def inspect_mf_shift(base_mfs, adapted_mfs, universe, names, scenario):
    print(f"\n{scenario}")

    for name in names:
        base_peak = mf_peak_x(base_mfs[name], universe)
        adapted_peak = mf_peak_x(adapted_mfs[name], universe)

        base_centroid = mf_centroid_x(base_mfs[name], universe)
        adapted_centroid = mf_centroid_x(adapted_mfs[name], universe)

        print(
            f"{name}: "
            f"peak {base_peak:.2f} -> {adapted_peak:.2f} "
            f"(delta={adapted_peak - base_peak:+.2f}), "
            f"centroid {base_centroid:.2f} -> {adapted_centroid:.2f} "
            f"(delta={adapted_centroid - base_centroid:+.2f})"
        )


rpm_mfs = [
    "rpm_low",
    "rpm_medium",
    "rpm_high",
]

torque_mfs = [
    "torque_low",
    "torque_medium",
    "torque_high",
]


inspect_mf_shift(
    evolving_mfs,
    sudden_adapted_mfs,
    rpm,
    rpm_mfs,
    "Sudden Drift — RPM"
)

inspect_mf_shift(
    evolving_mfs,
    sudden_adapted_mfs,
    torque,
    torque_mfs,
    "Sudden Drift — Torque"
)

inspect_mf_shift(
    evolving_mfs,
    gradual_adapted_mfs,
    rpm,
    rpm_mfs,
    "Gradual Drift — RPM"
)

inspect_mf_shift(
    evolving_mfs,
    gradual_adapted_mfs,
    torque,
    torque_mfs,
    "Gradual Drift — Torque"
)


# Verify all adapted MFs remain valid
for mfs, label in [
    (sudden_adapted_mfs, "Sudden"),
    (gradual_adapted_mfs, "Gradual"),
]:
    for name in rpm_mfs + torque_mfs:
        assert np.all(mfs[name] >= 0)
        assert np.all(mfs[name] <= 1)

print("\nMF value range [0, 1]: PASS")
print("MF geometry verification: COMPLETE")


=== PHASE 7.6 — VERIFY ADAPTED MF GEOMETRY ===

Sudden Drift — RPM
rpm_low: peak 1168.00 -> 1168.00 (delta=+0.00), centroid 1300.15 -> 1237.60 (delta=-62.54)
rpm_medium: peak 1505.07 -> 1370.93 (delta=-134.14), centroid 1551.33 -> 1418.43 (delta=-132.90)
rpm_high: peak 1881.68 -> 1749.27 (delta=-132.42), centroid 2310.15 -> 2177.06 (delta=-133.09)

Sudden Drift — Torque
torque_low: peak 3.80 -> 10.90 (delta=+7.10), centroid 18.48 -> 25.53 (delta=+7.06)
torque_medium: peak 40.04 -> 47.07 (delta=+7.03), centroid 40.00 -> 47.05 (delta=+7.05)
torque_high: peak 55.04 -> 62.14 (delta=+7.10), centroid 61.52 -> 64.94 (delta=+3.42)

Gradual Drift — RPM
rpm_low: peak 1168.00 -> 1168.00 (delta=+0.00), centroid 1300.15 -> 1235.30 (delta=-64.85)
rpm_medium: peak 1505.07 -> 1365.77 (delta=-139.30), centroid 1551.33 -> 1413.15 (delta=-138.19)
rpm_high: peak 1881.68 -> 1744.11 (delta=-137.58), centroid 2310.15 -> 2171.84 (delta=-138.31)

Gradual Drift — Torque
torque_low: peak 3.80 -> 11.70 (delta=+7.

## Phase 7.7 — Inspect Inference Function


In [8]:
print("=== PHASE 7.7 — INSPECT INFERENCE FUNCTION ===")

import inspect

print(inspect.signature(mamdani_inference))
print("\nSource:")
print(inspect.getsource(mamdani_inference))


=== PHASE 7.7 — INSPECT INFERENCE FUNCTION ===
(sample, return_details=False)

Source:
def mamdani_inference(sample, return_details=False):
    mu = compute_memberships(sample)
    rule_activations = {}
    implied_outputs = []
    
    for r_id, antecedents, consequent in rules:
        alpha = min(mu[var][term] for var, term in antecedents)
        rule_activations[r_id] = alpha
        implied_mf = np.fmin(alpha, anomaly_mfs[consequent])
        implied_outputs.append(implied_mf)
        
    aggregated_mf = np.zeros_like(anomaly_universe)
    for mf in implied_outputs:
        aggregated_mf = np.fmax(aggregated_mf, mf)
        
    if np.sum(aggregated_mf) == 0:
        score = 0.0
    else:
        score = fuzz.defuzz(anomaly_universe, aggregated_mf, "centroid")
        
    if return_details:
        return score, rule_activations, mu, aggregated_mf
    return score



## Phase 7.8 — Inspect Membership Function


In [9]:
print("=== PHASE 7.8 — INSPECT MEMBERSHIP FUNCTION ===")

import inspect

print(inspect.signature(compute_memberships))
print("\nSource:")
print(inspect.getsource(compute_memberships))


=== PHASE 7.8 — INSPECT MEMBERSHIP FUNCTION ===
(sample)

Source:
def compute_memberships(sample):
    mapping = {
        "air_temp":     sample["Air temperature [K]"],
        "process_temp": sample["Process temperature [K]"],
        "rpm":          sample["Rotational speed [rpm]"],
        "torque":       sample["Torque [Nm]"],
        "tool_wear":    sample["Tool wear [min]"],
    }
    m = {}
    for var_name, val in mapping.items():
        m[var_name] = {}
        for term, (m_type, params) in mf_params[var_name].items():
            m[var_name][term] = eval_mf(val, m_type, params)
    return m



## Phase 7.9 — Evolving Membership Computation


In [10]:
print("=== PHASE 7.9 — FIX EVOLVING MEMBERSHIP COMPUTATION ===")

evolving_universes = {
    "air_temp": air_temp,
    "process_temp": process_temp,
    "rpm": rpm,
    "torque": torque,
    "tool_wear": tool_wear,
}

evolving_mf_names = {
    "air_temp": ["air_low", "air_medium", "air_high"],
    "process_temp": ["process_low", "process_medium", "process_high"],
    "rpm": ["rpm_low", "rpm_medium", "rpm_high"],
    "torque": ["torque_low", "torque_medium", "torque_high"],
    "tool_wear": ["tool_wear_low", "tool_wear_medium", "tool_wear_high"],
}

def compute_memberships_evolving(sample, mfs):
    mapping = {
        "air_temp": sample["Air temperature [K]"],
        "process_temp": sample["Process temperature [K]"],
        "rpm": sample["Rotational speed [rpm]"],
        "torque": sample["Torque [Nm]"],
        "tool_wear": sample["Tool wear [min]"],
    }

    m = {}

    for var_name, value in mapping.items():
        universe = evolving_universes[var_name]
        m[var_name] = {}

        for mf_name in evolving_mf_names[var_name]:
            # Always extract the final term name:
            # air_low -> low
            # process_medium -> medium
            # rpm_high -> high
            # tool_wear_low -> low
            term = mf_name.rsplit("_", 1)[-1]

            m[var_name][term] = float(
                np.interp(
                    value,
                    universe,
                    mfs[mf_name]
                )
            )

    return m


sample = sudden_stream.iloc[1184]

mu_static = compute_memberships(sample)
mu_evolving = compute_memberships_evolving(sample, sudden_adapted_mfs)

print("\nSanity check — UDI:", sample["UDI"])

print("\nStatic RPM memberships:")
print(mu_static["rpm"])

print("\nEvolving RPM memberships:")
print(mu_evolving["rpm"])

print("\nStatic Torque memberships:")
print(mu_static["torque"])

print("\nEvolving Torque memberships:")
print(mu_evolving["torque"])

print("\nEvolving Tool-wear memberships:")
print(mu_evolving["tool_wear"])

# Verify expected term keys
expected_terms = {"LOW", "MEDIUM", "HIGH"}

for variable in mu_evolving:
    actual_terms = {term.upper() for term in mu_evolving[variable].keys()}
    assert actual_terms == expected_terms, (
        f"{variable}: unexpected membership keys {mu_evolving[variable].keys()}"
    )

# Verify membership range
for variable in mu_evolving:
    for term, value in mu_evolving[variable].items():
        assert 0.0 <= value <= 1.0, (
            f"Invalid membership: {variable}/{term} = {value}"
        )

print("\nMembership keys: PASS")
print("Membership values in [0, 1]: PASS")
print("Evolving membership computation: COMPLETE")


=== PHASE 7.9 — FIX EVOLVING MEMBERSHIP COMPUTATION ===

Sanity check — UDI: 9185

Static RPM memberships:
{'LOW': 0.0, 'MEDIUM': 0.9628378378378378, 'HIGH': 0.0}

Evolving RPM memberships:
{'low': 0.0, 'medium': 0.5138445945945944, 'high': 0.17107857142857158}

Static Torque memberships:
{'LOW': 0.0, 'MEDIUM': 0.6933333333333332, 'HIGH': 0.30666666666666675}

Evolving Torque memberships:
{'low': 0.16323333333333306, 'medium': 0.836766666666667, 'high': 0.0}

Evolving Tool-wear memberships:
{'low': 1.0, 'medium': 0.0, 'high': 0.0}

Membership keys: PASS
Membership values in [0, 1]: PASS
Evolving membership computation: COMPLETE


## Phase 7.10 — Buffer-only MF Shift Estimation


In [11]:
print("=== PHASE 7.10 — BUFFER-ONLY MF SHIFT ESTIMATION ===")

# Compatibility check for NumPy 2.x trapz / trapezoid
if not hasattr(np, "trapz"):
    np.trapz = np.trapezoid

def get_mf_centroid(universe, mf):
    area = np.trapz(mf, universe)
    if area == 0:
        return None
    return np.trapz(universe * mf, universe) / area


def estimate_buffer_shift(buffer_df, variable_key, sensor_column, current_mfs):
    universe = evolving_universes[variable_key]

    buffer_mean = buffer_df[sensor_column].mean()

    medium_mf_name = evolving_mf_names[variable_key][1]
    medium_centroid = get_mf_centroid(
        universe,
        current_mfs[medium_mf_name]
    )

    shift = buffer_mean - medium_centroid

    return {
        "buffer_mean": buffer_mean,
        "medium_centroid": medium_centroid,
        "estimated_shift": shift,
    }


# ---------------------------------------------------------
# Sudden drift
# ---------------------------------------------------------

sudden_detection = 1183
sudden_buffer = sudden_stream.iloc[
    sudden_detection + 1 :
    sudden_detection + 1 + ADAPTATION_WINDOW
].copy()

sudden_rpm_shift = estimate_buffer_shift(
    sudden_buffer,
    "rpm",
    "Rotational speed [rpm]",
    evolving_mfs
)

sudden_torque_shift = estimate_buffer_shift(
    sudden_buffer,
    "torque",
    "Torque [Nm]",
    evolving_mfs
)


# ---------------------------------------------------------
# Gradual drift
# ---------------------------------------------------------

gradual_detection = 1247
gradual_buffer = gradual_stream.iloc[
    gradual_detection + 1 :
    gradual_detection + 1 + ADAPTATION_WINDOW
].copy()

gradual_rpm_shift = estimate_buffer_shift(
    gradual_buffer,
    "rpm",
    "Rotational speed [rpm]",
    evolving_mfs
)

gradual_torque_shift = estimate_buffer_shift(
    gradual_buffer,
    "torque",
    "Torque [Nm]",
    evolving_mfs
)


# ---------------------------------------------------------
# Report
# ---------------------------------------------------------

print("\n--- Sudden Drift ---")

print(
    f"RPM    | buffer mean = {sudden_rpm_shift['buffer_mean']:.4f} | "
    f"MEDIUM centroid = {sudden_rpm_shift['medium_centroid']:.4f} | "
    f"shift = {sudden_rpm_shift['estimated_shift']:+.4f}"
)

print(
    f"Torque | buffer mean = {sudden_torque_shift['buffer_mean']:.4f} | "
    f"MEDIUM centroid = {sudden_torque_shift['medium_centroid']:.4f} | "
    f"shift = {sudden_torque_shift['estimated_shift']:+.4f}"
)


print("\n--- Gradual Drift ---")

print(
    f"RPM    | buffer mean = {gradual_rpm_shift['buffer_mean']:.4f} | "
    f"MEDIUM centroid = {gradual_rpm_shift['medium_centroid']:.4f} | "
    f"shift = {gradual_rpm_shift['estimated_shift']:+.4f}"
)

print(
    f"Torque | buffer mean = {gradual_torque_shift['buffer_mean']:.4f} | "
    f"MEDIUM centroid = {gradual_torque_shift['medium_centroid']:.4f} | "
    f"shift = {gradual_torque_shift['estimated_shift']:+.4f}"
)


# ---------------------------------------------------------
# Basic sanity checks
# ---------------------------------------------------------

assert len(sudden_buffer) == ADAPTATION_WINDOW
assert len(gradual_buffer) == ADAPTATION_WINDOW

assert np.isfinite(sudden_rpm_shift["estimated_shift"])
assert np.isfinite(sudden_torque_shift["estimated_shift"])
assert np.isfinite(gradual_rpm_shift["estimated_shift"])
assert np.isfinite(gradual_torque_shift["estimated_shift"])

print("\nBuffer size check: PASS")
print("Shift values finite: PASS")
print("Buffer-only adaptation estimation: COMPLETE")


=== PHASE 7.10 — BUFFER-ONLY MF SHIFT ESTIMATION ===

--- Sudden Drift ---
RPM    | buffer mean = 1395.1200 | MEDIUM centroid = 1551.3342 | shift = -156.2142
Torque | buffer mean = 47.3015 | MEDIUM centroid = 40.0000 | shift = +7.3015

--- Gradual Drift ---
RPM    | buffer mean = 1389.3750 | MEDIUM centroid = 1551.3342 | shift = -161.9592
Torque | buffer mean = 48.0530 | MEDIUM centroid = 40.0000 | shift = +8.0530

Buffer size check: PASS
Shift values finite: PASS
Buffer-only adaptation estimation: COMPLETE


## Phase 7.11 — Apply Buffer-only MF Adaptation


In [12]:
print("=== PHASE 7.11 — APPLY BUFFER-ONLY MF ADAPTATION ===")


def apply_mf_shift(mfs, variable_key, shift):
    adapted = {
        name: mf.copy()
        for name, mf in mfs.items()
    }

    universe = evolving_universes[variable_key]

    for mf_name in evolving_mf_names[variable_key]:
        adapted[mf_name] = np.interp(
            universe - shift,
            universe,
            mfs[mf_name],
            left=0.0,
            right=0.0
        )

    return adapted


# ---------------------------------------------------------
# Sudden drift adaptation
# ---------------------------------------------------------

sudden_buffer_adapted_mfs = {
    name: mf.copy()
    for name, mf in evolving_mfs.items()
}

sudden_buffer_adapted_mfs = apply_mf_shift(
    sudden_buffer_adapted_mfs,
    "rpm",
    sudden_rpm_shift["estimated_shift"]
)

sudden_buffer_adapted_mfs = apply_mf_shift(
    sudden_buffer_adapted_mfs,
    "torque",
    sudden_torque_shift["estimated_shift"]
)


# ---------------------------------------------------------
# Gradual drift adaptation
# ---------------------------------------------------------

gradual_buffer_adapted_mfs = {
    name: mf.copy()
    for name, mf in evolving_mfs.items()
}

gradual_buffer_adapted_mfs = apply_mf_shift(
    gradual_buffer_adapted_mfs,
    "rpm",
    gradual_rpm_shift["estimated_shift"]
)

gradual_buffer_adapted_mfs = apply_mf_shift(
    gradual_buffer_adapted_mfs,
    "torque",
    gradual_torque_shift["estimated_shift"]
)


# ---------------------------------------------------------
# Verify protected variables remain unchanged
# ---------------------------------------------------------

protected_variables = [
    "air_temp",
    "process_temp",
    "tool_wear",
]

for variable_key in protected_variables:
    for mf_name in evolving_mf_names[variable_key]:
        assert np.array_equal(
            evolving_mfs[mf_name],
            sudden_buffer_adapted_mfs[mf_name]
        )
        assert np.array_equal(
            evolving_mfs[mf_name],
            gradual_buffer_adapted_mfs[mf_name]
        )

print("\nProtected Air/Process/Tool-wear MFs: PASS")


# ---------------------------------------------------------
# Verify adapted variables actually changed
# ---------------------------------------------------------

for mf_name in (
    evolving_mf_names["rpm"]
    + evolving_mf_names["torque"]
):
    assert not np.array_equal(
        evolving_mfs[mf_name],
        sudden_buffer_adapted_mfs[mf_name]
    )

    assert not np.array_equal(
        evolving_mfs[mf_name],
        gradual_buffer_adapted_mfs[mf_name]
    )

print("RPM/Torque MFs adapted: PASS")


# ---------------------------------------------------------
# Verify membership range
# ---------------------------------------------------------

for adapted_mfs in [
    sudden_buffer_adapted_mfs,
    gradual_buffer_adapted_mfs,
]:
    for mf_name, mf in adapted_mfs.items():
        assert np.all(np.isfinite(mf))
        assert np.all(mf >= 0.0)
        assert np.all(mf <= 1.0)


print("All adapted MF values in [0, 1]: PASS")


# ---------------------------------------------------------
# Verify rule structure is still unchanged
# ---------------------------------------------------------

assert len(rules) == 12

print("Static rule structure preserved: PASS")
print("\nBuffer-only MF adaptation: COMPLETE")


=== PHASE 7.11 — APPLY BUFFER-ONLY MF ADAPTATION ===

Protected Air/Process/Tool-wear MFs: PASS
RPM/Torque MFs adapted: PASS
All adapted MF values in [0, 1]: PASS
Static rule structure preserved: PASS

Buffer-only MF adaptation: COMPLETE


## Phase 7.12 — Evolving Mamdani Inference


In [13]:
print("=== PHASE 7.12 — EVOLVING MAMDANI INFERENCE ===")


def mamdani_inference_evolving(sample, adapted_mfs, return_details=False):
    mu = compute_memberships_evolving(sample, adapted_mfs)

    rule_activations = {}
    implied_outputs = []

    for r_id, antecedents, consequent in rules:

        activation_values = []

        for var_name, term in antecedents:
            # Normalize rule term and membership term
            normalized_term = term.lower()

            activation_values.append(
                mu[var_name][normalized_term]
            )

        alpha = min(activation_values)

        rule_activations[r_id] = alpha

        implied_mf = np.fmin(
            alpha,
            anomaly_mfs[consequent]
        )

        implied_outputs.append(implied_mf)

    aggregated_mf = np.zeros_like(anomaly_universe)

    for mf in implied_outputs:
        aggregated_mf = np.fmax(
            aggregated_mf,
            mf
        )

    if np.sum(aggregated_mf) == 0:
        score = 0.0
    else:
        score = fuzz.defuzz(
            anomaly_universe,
            aggregated_mf,
            "centroid"
        )

    if return_details:
        return (
            score,
            rule_activations,
            mu,
            aggregated_mf
        )

    return score


# ---------------------------------------------------------
# Sanity check on first post-detection sample
# ---------------------------------------------------------

sample = sudden_stream.iloc[1184]

static_score = mamdani_inference(sample)

evolving_score = mamdani_inference_evolving(
    sample,
    sudden_buffer_adapted_mfs
)

print("\nSanity check — UDI:", sample["UDI"])

print(f"Static score   : {static_score:.6f}")
print(f"Evolving score : {evolving_score:.6f}")
print(f"Score delta    : {evolving_score - static_score:+.6f}")


# ---------------------------------------------------------
# Basic validity checks
# ---------------------------------------------------------

assert np.isfinite(evolving_score)
assert 0.0 <= evolving_score <= 1.0

print("\nEvolving anomaly score in [0, 1]: PASS")
print("Evolving Mamdani inference: COMPLETE")


=== PHASE 7.12 — EVOLVING MAMDANI INFERENCE ===

Sanity check — UDI: 9185
Static score   : 0.209696
Evolving score : 0.223914
Score delta    : +0.014218

Evolving anomaly score in [0, 1]: PASS
Evolving Mamdani inference: COMPLETE


## Phase 7.13 — Static vs Evolving Diagnostic on Adaptation Buffer


In [14]:
print("=== PHASE 7.13 — STATIC VS EVOLVING DIAGNOSTIC ===")


def compare_static_evolving(buffer_df, adapted_mfs, sample_indices):
    rows = []

    for idx in sample_indices:
        sample = buffer_df.iloc[idx]

        static_score = mamdani_inference(sample)

        evolving_score = mamdani_inference_evolving(
            sample,
            adapted_mfs
        )

        rows.append({
            "stream_index": sample.name,
            "UDI": sample["UDI"],
            "static_score": static_score,
            "evolving_score": evolving_score,
            "delta": evolving_score - static_score,
        })

    return pd.DataFrame(rows)


# ---------------------------------------------------------
# Sudden adaptation buffer
# ---------------------------------------------------------

sudden_sample_indices = list(range(10)) + list(
    range(
        ADAPTATION_WINDOW - 10,
        ADAPTATION_WINDOW
    )
)

sudden_diagnostic = compare_static_evolving(
    sudden_buffer,
    sudden_buffer_adapted_mfs,
    sudden_sample_indices
)


# ---------------------------------------------------------
# Gradual adaptation buffer
# ---------------------------------------------------------

gradual_diagnostic = compare_static_evolving(
    gradual_buffer,
    gradual_buffer_adapted_mfs,
    sudden_sample_indices
)


print("\n--- Sudden Drift Buffer ---")
print(sudden_diagnostic.to_string(index=False))

print("\n--- Gradual Drift Buffer ---")
print(gradual_diagnostic.to_string(index=False))


# ---------------------------------------------------------
# Basic validity checks
# ---------------------------------------------------------

for diagnostic in [
    sudden_diagnostic,
    gradual_diagnostic,
]:
    assert len(diagnostic) == 20
    assert np.all(np.isfinite(diagnostic["static_score"]))
    assert np.all(np.isfinite(diagnostic["evolving_score"]))
    assert np.all(
        (diagnostic["static_score"] >= 0.0)
        & (diagnostic["static_score"] <= 1.0)
    )
    assert np.all(
        (diagnostic["evolving_score"] >= 0.0)
        & (diagnostic["evolving_score"] <= 1.0)
    )

print("\nDiagnostic score validity: PASS")
print("Static vs Evolving diagnostic: COMPLETE")


=== PHASE 7.13 — STATIC VS EVOLVING DIAGNOSTIC ===

--- Sudden Drift Buffer ---
 stream_index  UDI  static_score  evolving_score         delta
         1184 9185      0.209696        0.223914  1.421814e-02
         1185 9186      0.500000        0.376217 -1.237834e-01
         1186 9187      0.204167        0.217081  1.291417e-02
         1187 9188      0.500000        0.411483 -8.851686e-02
         1188 9189      0.493944        0.197261 -2.966824e-01
         1189 9190      0.213789        0.219497  5.708594e-03
         1190 9191      0.500000        0.213713 -2.862867e-01
         1191 9192      0.209696        0.194445 -1.525087e-02
         1192 9193      0.321036        0.205876 -1.151600e-01
         1193 9194      0.426090        0.199624 -2.264655e-01
         1374 9375      0.500000        0.269743 -2.302567e-01
         1375 9376      0.500000        0.500000 -1.720846e-15
         1376 9377      0.208642        0.219396  1.075322e-02
         1377 9378      0.500000      

## Phase 7.14 — Post-Adaptation Score Evaluation


In [15]:
print("=== PHASE 7.14 — POST-ADAPTATION SCORE EVALUATION ===")


def evaluate_post_adaptation(
    stream_df,
    adaptation_end,
    adapted_mfs
):
    evaluation_start = adaptation_end + 1

    rows = []

    for idx in range(evaluation_start, len(stream_df)):
        sample = stream_df.iloc[idx]

        static_score = mamdani_inference(sample)

        evolving_score = mamdani_inference_evolving(
            sample,
            adapted_mfs
        )

        rows.append({
            "stream_index": idx,
            "UDI": sample["UDI"],
            "label": int(sample["Machine failure"]),
            "static_score": static_score,
            "evolving_score": evolving_score,
        })

    return pd.DataFrame(rows)


# ---------------------------------------------------------
# Sudden
# ---------------------------------------------------------

sudden_adaptation_end = (
    sudden_detection + ADAPTATION_WINDOW
)

sudden_post_adaptation = evaluate_post_adaptation(
    sudden_stream,
    sudden_adaptation_end,
    sudden_buffer_adapted_mfs
)


# ---------------------------------------------------------
# Gradual
# ---------------------------------------------------------

gradual_adaptation_end = (
    gradual_detection + ADAPTATION_WINDOW
)

gradual_post_adaptation = evaluate_post_adaptation(
    gradual_stream,
    gradual_adaptation_end,
    gradual_buffer_adapted_mfs
)


# ---------------------------------------------------------
# Summary
# ---------------------------------------------------------

print("\n--- Sudden Drift ---")
print(
    f"Evaluation range: "
    f"{sudden_post_adaptation['stream_index'].min()} → "
    f"{sudden_post_adaptation['stream_index'].max()}"
)
print(f"Samples: {len(sudden_post_adaptation)}")
print(
    f"Static mean score   : "
    f"{sudden_post_adaptation['static_score'].mean():.6f}"
)
print(
    f"Evolving mean score : "
    f"{sudden_post_adaptation['evolving_score'].mean():.6f}"
)


print("\n--- Gradual Drift ---")
print(
    f"Evaluation range: "
    f"{gradual_post_adaptation['stream_index'].min()} → "
    f"{gradual_post_adaptation['stream_index'].max()}"
)
print(f"Samples: {len(gradual_post_adaptation)}")
print(
    f"Static mean score   : "
    f"{gradual_post_adaptation['static_score'].mean():.6f}"
)
print(
    f"Evolving mean score : "
    f"{gradual_post_adaptation['evolving_score'].mean():.6f}"
)


# ---------------------------------------------------------
# Validity checks
# ---------------------------------------------------------

assert len(sudden_post_adaptation) == (
    len(sudden_stream)
    - sudden_adaptation_end
    - 1
)

assert len(gradual_post_adaptation) == (
    len(gradual_stream)
    - gradual_adaptation_end
    - 1
)


for df in [
    sudden_post_adaptation,
    gradual_post_adaptation,
]:
    assert np.all(np.isfinite(df["static_score"]))
    assert np.all(np.isfinite(df["evolving_score"]))

    assert np.all(
        (df["static_score"] >= 0.0)
        & (df["static_score"] <= 1.0)
    )

    assert np.all(
        (df["evolving_score"] >= 0.0)
        & (df["evolving_score"] <= 1.0)
    )


print("\nPost-adaptation sample boundary: PASS")
print("Score validity: PASS")
print("Post-adaptation evaluation: COMPLETE")


=== PHASE 7.14 — POST-ADAPTATION SCORE EVALUATION ===

--- Sudden Drift ---
Evaluation range: 1384 → 1999
Samples: 616
Static mean score   : 0.462804
Evolving mean score : 0.325958

--- Gradual Drift ---
Evaluation range: 1448 → 1999
Samples: 552
Static mean score   : 0.464662
Evolving mean score : 0.315719

Post-adaptation sample boundary: PASS
Score validity: PASS
Post-adaptation evaluation: COMPLETE


## Phase 7.15 — Frozen-Threshold Metrics


In [16]:
print("=== PHASE 7.15 — FROZEN-THRESHOLD METRICS ===")


def compute_frozen_metrics(df, score_column, threshold=TAU_STATIC):
    y_true = df["label"].to_numpy(dtype=int)
    y_pred = (
        df[score_column].to_numpy(dtype=float) >= threshold
    ).astype(int)

    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))

    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0.0
    )

    recall = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0.0
    )

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    return {
        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
    }


# ---------------------------------------------------------
# Compute metrics
# ---------------------------------------------------------

sudden_static_metrics = compute_frozen_metrics(
    sudden_post_adaptation,
    "static_score"
)

sudden_evolving_metrics = compute_frozen_metrics(
    sudden_post_adaptation,
    "evolving_score"
)

gradual_static_metrics = compute_frozen_metrics(
    gradual_post_adaptation,
    "static_score"
)

gradual_evolving_metrics = compute_frozen_metrics(
    gradual_post_adaptation,
    "evolving_score"
)


# ---------------------------------------------------------
# Display
# ---------------------------------------------------------

print(f"\nFrozen threshold: TAU = {TAU_STATIC:.2f}")

print("\n--- Sudden Drift ---")

print("\nStatic:")
for key, value in sudden_static_metrics.items():
    print(f"{key:>10}: {value:.6f}" if isinstance(value, float)
          else f"{key:>10}: {value}")

print("\nEvolving:")
for key, value in sudden_evolving_metrics.items():
    print(f"{key:>10}: {value:.6f}" if isinstance(value, float)
          else f"{key:>10}: {value}")


print("\n--- Gradual Drift ---")

print("\nStatic:")
for key, value in gradual_static_metrics.items():
    print(f"{key:>10}: {value:.6f}" if isinstance(value, float)
          else f"{key:>10}: {value}")

print("\nEvolving:")
for key, value in gradual_evolving_metrics.items():
    print(f"{key:>10}: {value:.6f}" if isinstance(value, float)
          else f"{key:>10}: {value}")


# ---------------------------------------------------------
# Validity checks
# ---------------------------------------------------------

for metrics in [
    sudden_static_metrics,
    sudden_evolving_metrics,
    gradual_static_metrics,
    gradual_evolving_metrics,
]:
    assert metrics["TP"] >= 0
    assert metrics["TN"] >= 0
    assert metrics["FP"] >= 0
    assert metrics["FN"] >= 0

    assert 0.0 <= metrics["Precision"] <= 1.0
    assert 0.0 <= metrics["Recall"] <= 1.0
    assert 0.0 <= metrics["F1"] <= 1.0


# Confusion matrix totals must equal evaluation size
assert (
    sudden_evolving_metrics["TP"]
    + sudden_evolving_metrics["TN"]
    + sudden_evolving_metrics["FP"]
    + sudden_evolving_metrics["FN"]
    == len(sudden_post_adaptation)
)

assert (
    gradual_evolving_metrics["TP"]
    + gradual_evolving_metrics["TN"]
    + gradual_evolving_metrics["FP"]
    + gradual_evolving_metrics["FN"]
    == len(gradual_post_adaptation)
)


print("\nMetric range checks: PASS")
print("Confusion matrix totals: PASS")
print("Frozen-threshold metrics: COMPLETE")


=== PHASE 7.15 — FROZEN-THRESHOLD METRICS ===

Frozen threshold: TAU = 0.67

--- Sudden Drift ---

Static:
        TP: 11
        TN: 537
        FP: 64
        FN: 4
 Precision: 0.146667
    Recall: 0.733333
        F1: 0.244444

Evolving:
        TP: 8
        TN: 595
        FP: 6
        FN: 7
 Precision: 0.571429
    Recall: 0.533333
        F1: 0.551724

--- Gradual Drift ---

Static:
        TP: 10
        TN: 483
        FP: 55
        FN: 4
 Precision: 0.153846
    Recall: 0.714286
        F1: 0.253165

Evolving:
        TP: 6
        TN: 535
        FP: 3
        FN: 8
 Precision: 0.666667
    Recall: 0.428571
        F1: 0.521739

Metric range checks: PASS
Confusion matrix totals: PASS
Frozen-threshold metrics: COMPLETE


## Phase 7.16 — Post-Adaptation Label Distribution


In [17]:
print("=== PHASE 7.16 — POST-ADAPTATION LABEL DISTRIBUTION ===")


def summarize_labels(df, scenario_name):
    total = len(df)

    failure_count = int(df["label"].sum())
    normal_count = total - failure_count

    failure_rate = failure_count / total if total > 0 else 0.0

    print(f"\n--- {scenario_name} ---")
    print(f"Total samples : {total}")
    print(f"Normal        : {normal_count}")
    print(f"Failure       : {failure_count}")
    print(f"Failure rate  : {failure_rate:.6f}")

    return {
        "total": total,
        "normal": normal_count,
        "failure": failure_count,
        "failure_rate": failure_rate,
    }


sudden_post_labels = summarize_labels(
    sudden_post_adaptation,
    "Sudden Drift — Post Adaptation"
)

gradual_post_labels = summarize_labels(
    gradual_post_adaptation,
    "Gradual Drift — Post Adaptation"
)


# ---------------------------------------------------------
# Consistency checks
# ---------------------------------------------------------

assert (
    sudden_post_labels["normal"]
    + sudden_post_labels["failure"]
    == sudden_post_labels["total"]
)

assert (
    gradual_post_labels["normal"]
    + gradual_post_labels["failure"]
    == gradual_post_labels["total"]
)

# Labels must remain unchanged across scenarios.
assert (
    sudden_post_adaptation["label"].to_numpy()
    == sudden_stream.loc[
        sudden_post_adaptation["stream_index"],
        "Machine failure"
    ].to_numpy()
).all()

assert (
    gradual_post_adaptation["label"].to_numpy()
    == gradual_stream.loc[
        gradual_post_adaptation["stream_index"],
        "Machine failure"
    ].to_numpy()
).all()


print("\nLabel-count consistency: PASS")
print("Scenario label integrity: PASS")
print("Post-adaptation label distribution: COMPLETE")


=== PHASE 7.16 — POST-ADAPTATION LABEL DISTRIBUTION ===

--- Sudden Drift — Post Adaptation ---
Total samples : 616
Normal        : 601
Failure       : 15
Failure rate  : 0.024351

--- Gradual Drift — Post Adaptation ---
Total samples : 552
Normal        : 538
Failure       : 14
Failure rate  : 0.025362

Label-count consistency: PASS
Scenario label integrity: PASS
Post-adaptation label distribution: COMPLETE


## Phase 7.17 — Post-Adaptation Additional Metrics


In [18]:
# PHASE 7.17 — POST-ADAPTATION ADDITIONAL METRICS

print("=== PHASE 7.17 — POST-ADAPTATION ADDITIONAL METRICS ===")


def compute_additional_metrics(tp, tn, fp, fn):
    total = tp + tn + fp + fn

    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    balanced_accuracy = (recall + specificity) / 2

    return {
        "Total": total,
        "FPR": fpr,
        "FNR": fnr,
        "Specificity": specificity,
        "Balanced Accuracy": balanced_accuracy,
    }


# Metrics from Cell 7.15
sudden_static_additional = compute_additional_metrics(
    tp=11, tn=537, fp=64, fn=4
)

sudden_evolving_additional = compute_additional_metrics(
    tp=8, tn=595, fp=6, fn=7
)

gradual_static_additional = compute_additional_metrics(
    tp=10, tn=483, fp=55, fn=4
)

gradual_evolving_additional = compute_additional_metrics(
    tp=6, tn=535, fp=3, fn=8
)


results = {
    "Sudden — Static": sudden_static_additional,
    "Sudden — Evolving": sudden_evolving_additional,
    "Gradual — Static": gradual_static_additional,
    "Gradual — Evolving": gradual_evolving_additional,
}


for name, metrics in results.items():
    print(f"\n--- {name} ---")
    print(f"Total samples       : {metrics['Total']}")
    print(f"FPR                 : {metrics['FPR']:.6f}")
    print(f"FNR                 : {metrics['FNR']:.6f}")
    print(f"Specificity         : {metrics['Specificity']:.6f}")
    print(f"Balanced Accuracy   : {metrics['Balanced Accuracy']:.6f}")


# Basic validity checks
for name, metrics in results.items():
    assert metrics["Total"] > 0
    assert 0.0 <= metrics["FPR"] <= 1.0
    assert 0.0 <= metrics["FNR"] <= 1.0
    assert 0.0 <= metrics["Specificity"] <= 1.0
    assert 0.0 <= metrics["Balanced Accuracy"] <= 1.0

print("\nMetric range checks: PASS")
print("Post-adaptation additional metrics: COMPLETE")


=== PHASE 7.17 — POST-ADAPTATION ADDITIONAL METRICS ===

--- Sudden — Static ---
Total samples       : 616
FPR                 : 0.106489
FNR                 : 0.266667
Specificity         : 0.893511
Balanced Accuracy   : 0.813422

--- Sudden — Evolving ---
Total samples       : 616
FPR                 : 0.009983
FNR                 : 0.466667
Specificity         : 0.990017
Balanced Accuracy   : 0.761675

--- Gradual — Static ---
Total samples       : 552
FPR                 : 0.102230
FNR                 : 0.285714
Specificity         : 0.897770
Balanced Accuracy   : 0.806028

--- Gradual — Evolving ---
Total samples       : 552
FPR                 : 0.005576
FNR                 : 0.571429
Specificity         : 0.994424
Balanced Accuracy   : 0.711498

Metric range checks: PASS
Post-adaptation additional metrics: COMPLETE


## Phase 7.18 — Overall Streaming Performance


In [19]:
# PHASE 7.18 — OVERALL STREAMING PERFORMANCE

print("=== PHASE 7.18 — OVERALL STREAMING PERFORMANCE ===")


def build_overall_streaming_result(
    stream_df,
    post_adaptation_df,
    adaptation_end,
    score_column,
    scenario_name,
    model_name
):
    # Static scores before adaptation end
    pre_adaptation = stream_df.loc[
        stream_df.index <= adaptation_end
    ].copy()

    pre_scores = [
        mamdani_inference(row)
        for _, row in pre_adaptation.iterrows()
    ]

    # Evolving/static scores after adaptation
    post_scores = post_adaptation_df[score_column].to_numpy()

    # Combine in chronological order
    labels = np.concatenate([
        pre_adaptation["Machine failure"].to_numpy(),
        post_adaptation_df["label"].to_numpy()
    ])

    scores = np.concatenate([
        np.asarray(pre_scores, dtype=float),
        post_scores.astype(float)
    ])

    assert len(scores) == len(stream_df)
    assert len(labels) == len(stream_df)
    assert np.all(np.isfinite(scores))
    assert np.all((scores >= 0.0) & (scores <= 1.0))

    predictions = (scores >= TAU_STATIC).astype(int)

    tp = int(np.sum((predictions == 1) & (labels == 1)))
    tn = int(np.sum((predictions == 0) & (labels == 0)))
    fp = int(np.sum((predictions == 1) & (labels == 0)))
    fn = int(np.sum((predictions == 0) & (labels == 1)))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    additional = compute_additional_metrics(tp, tn, fp, fn)

    result = {
        "Scenario": scenario_name,
        "Model": model_name,
        "Total": len(scores),
        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "FPR": additional["FPR"],
        "FNR": additional["FNR"],
        "Specificity": additional["Specificity"],
        "Balanced Accuracy": additional["Balanced Accuracy"],
    }

    return result


# ---------------------------------------------------------
# Sudden Drift
# ---------------------------------------------------------

sudden_overall_static = build_overall_streaming_result(
    stream_df=sudden_stream,
    post_adaptation_df=sudden_post_adaptation,
    adaptation_end=sudden_adaptation_end,
    score_column="static_score",
    scenario_name="Sudden Drift",
    model_name="Static"
)

sudden_overall_evolving = build_overall_streaming_result(
    stream_df=sudden_stream,
    post_adaptation_df=sudden_post_adaptation,
    adaptation_end=sudden_adaptation_end,
    score_column="evolving_score",
    scenario_name="Sudden Drift",
    model_name="Evolving"
)


# ---------------------------------------------------------
# Gradual Drift
# ---------------------------------------------------------

gradual_overall_static = build_overall_streaming_result(
    stream_df=gradual_stream,
    post_adaptation_df=gradual_post_adaptation,
    adaptation_end=gradual_adaptation_end,
    score_column="static_score",
    scenario_name="Gradual Drift",
    model_name="Static"
)

gradual_overall_evolving = build_overall_streaming_result(
    stream_df=gradual_stream,
    post_adaptation_df=gradual_post_adaptation,
    adaptation_end=gradual_adaptation_end,
    score_column="evolving_score",
    scenario_name="Gradual Drift",
    model_name="Evolving"
)


overall_results = [
    sudden_overall_static,
    sudden_overall_evolving,
    gradual_overall_static,
    gradual_overall_evolving,
]


# ---------------------------------------------------------
# Display
# ---------------------------------------------------------

for result in overall_results:
    print(f"\n--- {result['Scenario']} — {result['Model']} ---")
    print(f"Total samples       : {result['Total']}")
    print(f"TP                  : {result['TP']}")
    print(f"TN                  : {result['TN']}")
    print(f"FP                  : {result['FP']}")
    print(f"FN                  : {result['FN']}")
    print(f"Precision           : {result['Precision']:.6f}")
    print(f"Recall              : {result['Recall']:.6f}")
    print(f"F1                  : {result['F1']:.6f}")
    print(f"FPR                 : {result['FPR']:.6f}")
    print(f"FNR                 : {result['FNR']:.6f}")
    print(f"Specificity         : {result['Specificity']:.6f}")
    print(f"Balanced Accuracy   : {result['Balanced Accuracy']:.6f}")


# ---------------------------------------------------------
# Integrity checks
# ---------------------------------------------------------

for result in overall_results:
    assert result["Total"] == 2000
    assert result["TP"] + result["TN"] + result["FP"] + result["FN"] == 2000

    for metric_name in [
        "Precision",
        "Recall",
        "F1",
        "FPR",
        "FNR",
        "Specificity",
        "Balanced Accuracy",
    ]:
        assert 0.0 <= result[metric_name] <= 1.0

print("\nOverall confusion-matrix consistency: PASS")
print("Overall metric range checks: PASS")
print("Overall streaming performance: COMPLETE")


=== PHASE 7.18 — OVERALL STREAMING PERFORMANCE ===

--- Sudden Drift — Static ---
Total samples       : 2000
TP                  : 18
TN                  : 1847
FP                  : 114
FN                  : 21
Precision           : 0.136364
Recall              : 0.461538
F1                  : 0.210526
FPR                 : 0.058134
FNR                 : 0.538462
Specificity         : 0.941866
Balanced Accuracy   : 0.701702

--- Sudden Drift — Evolving ---
Total samples       : 2000
TP                  : 15
TN                  : 1905
FP                  : 56
FN                  : 24
Precision           : 0.211268
Recall              : 0.384615
F1                  : 0.272727
FPR                 : 0.028557
FNR                 : 0.615385
Specificity         : 0.971443
Balanced Accuracy   : 0.678029

--- Gradual Drift — Static ---
Total samples       : 2000
TP                  : 18
TN                  : 1848
FP                  : 113
FN                  : 21
Precision           : 0.137405

## Phase 7.19 — Overall Performance Delta


In [20]:
# PHASE 7.19 — OVERALL PERFORMANCE DELTA

print("=== PHASE 7.19 — OVERALL PERFORMANCE DELTA ===")


def compute_performance_delta(static_res, evolving_res):
    delta = {
        "Scenario": static_res["Scenario"],
        "Delta TP": evolving_res["TP"] - static_res["TP"],
        "Delta TN": evolving_res["TN"] - static_res["TN"],
        "Delta FP": evolving_res["FP"] - static_res["FP"],
        "Delta FN": evolving_res["FN"] - static_res["FN"],
        "Delta Precision": evolving_res["Precision"] - static_res["Precision"],
        "Delta Recall": evolving_res["Recall"] - static_res["Recall"],
        "Delta F1": evolving_res["F1"] - static_res["F1"],
        "Delta FPR": evolving_res["FPR"] - static_res["FPR"],
        "Relative FPR Reduction (%)": (
            (static_res["FPR"] - evolving_res["FPR"]) / static_res["FPR"] * 100.0
            if static_res["FPR"] > 0
            else 0.0
        ),
        "Delta FNR": evolving_res["FNR"] - static_res["FNR"],
        "Delta Specificity": evolving_res["Specificity"] - static_res["Specificity"],
        "Delta Balanced Accuracy": (
            evolving_res["Balanced Accuracy"] - static_res["Balanced Accuracy"]
        ),
    }
    return delta


sudden_delta = compute_performance_delta(
    sudden_overall_static, sudden_overall_evolving
)
gradual_delta = compute_performance_delta(
    gradual_overall_static, gradual_overall_evolving
)

delta_results = [sudden_delta, gradual_delta]

for res in delta_results:
    print(f"\n--- {res['Scenario']} Delta (Evolving vs Static) ---")
    print(f"Delta TP                    : {res['Delta TP']:+d}")
    print(f"Delta TN                    : {res['Delta TN']:+d}")
    print(f"Delta FP (False Alarms)     : {res['Delta FP']:+d}")
    print(f"Delta FN (Missed Failures)  : {res['Delta FN']:+d}")
    print(f"Delta Precision             : {res['Delta Precision']:+.6f}")
    print(f"Delta Recall                : {res['Delta Recall']:+.6f}")
    print(f"Delta F1                    : {res['Delta F1']:+.6f}")
    print(f"Delta FPR                   : {res['Delta FPR']:+.6f}")
    print(f"Relative FPR Reduction      : {res['Relative FPR Reduction (%)']:.2f}%")
    print(f"Delta FNR                   : {res['Delta FNR']:+.6f}")
    print(f"Delta Specificity           : {res['Delta Specificity']:+.6f}")
    print(f"Delta Balanced Accuracy     : {res['Delta Balanced Accuracy']:+.6f}")


# ---------------------------------------------------------
# Integrity checks
# ---------------------------------------------------------

for res in delta_results:
    # Delta of counts must sum to 0
    assert (
        res["Delta TP"] + res["Delta TN"] + res["Delta FP"] + res["Delta FN"] == 0
    )
    # Signs of complementary metrics must be strictly opposite
    assert np.isclose(res["Delta FPR"], -res["Delta Specificity"])
    assert np.isclose(res["Delta FNR"], -res["Delta Recall"])

print("\nDelta mathematical consistency: PASS")
print("Overall performance delta: COMPLETE")


=== PHASE 7.19 — OVERALL PERFORMANCE DELTA ===

--- Sudden Drift Delta (Evolving vs Static) ---
Delta TP                    : -3
Delta TN                    : +58
Delta FP (False Alarms)     : -58
Delta FN (Missed Failures)  : +3
Delta Precision             : +0.074904
Delta Recall                : -0.076923
Delta F1                    : +0.062201
Delta FPR                   : -0.029577
Relative FPR Reduction      : 50.88%
Delta FNR                   : +0.076923
Delta Specificity           : +0.029577
Delta Balanced Accuracy     : -0.023673

--- Gradual Drift Delta (Evolving vs Static) ---
Delta TP                    : -4
Delta TN                    : +52
Delta FP (False Alarms)     : -52
Delta FN (Missed Failures)  : +4
Delta Precision             : +0.049262
Delta Recall                : -0.102564
Delta F1                    : +0.033849
Delta FPR                   : -0.026517
Relative FPR Reduction      : 46.02%
Delta FNR                   : +0.102564
Delta Specificity           : +0